In [14]:
import pandas as pd
import os

data_dir = "/Users/larasch/Documents/UCB_postdoc/Research/Mexico_heat_vulnerability/data/death_data"

period_files = {
    "1998_2002": "combined_1998_2002.csv",
    "2003_2007": "combined_2003_2007.csv",
    "2008_2012": "combined_2008_2012.csv",
    "2013_2017": "combined_2013_2017.csv",
    "2018_2022": "combined_2018_2022.csv"
}

output_dir = os.path.join(data_dir, "summaries")
os.makedirs(output_dir, exist_ok=True)

group_vars = ["date_of_occurrence", "ENT_OCURR", "MUN_OCURR"]

for period, filename in period_files.items():
    print(f"\nProcessing {period}...")

    file_path = os.path.join(data_dir, filename)
    df = pd.read_csv(file_path, low_memory=False)

    # Fix dates
    df["date_of_occurrence"] = pd.to_datetime(df["date_of_occurrence"], errors="coerce")

    df = df.reset_index(drop=True)   # remove existing index
    df.index.name = None             # remove index name
    df = df.loc[:, ~df.columns.duplicated()]   # drop duplicate columns

    # Numeric columns to sum
    numeric_cols = [
    c for c in df.select_dtypes(include="number").columns
    if c not in group_vars]

    # Group + sum
    grouped = df.groupby(group_vars, dropna=False)[numeric_cols].sum(min_count=1)

    # Reset index SAFELY
    summary = grouped.reset_index()  # no duplicates now

    # Save output
    out_path = os.path.join(output_dir, f"combined_sum_{period}.csv")
    summary.to_csv(out_path, index=False)

    print(f"Saved → {out_path}")



Processing 1998_2002...
Saved → /Users/larasch/Documents/UCB_postdoc/Research/Mexico_heat_vulnerability/data/death_data/summaries/combined_sum_1998_2002.csv

Processing 2003_2007...
Saved → /Users/larasch/Documents/UCB_postdoc/Research/Mexico_heat_vulnerability/data/death_data/summaries/combined_sum_2003_2007.csv

Processing 2008_2012...
Saved → /Users/larasch/Documents/UCB_postdoc/Research/Mexico_heat_vulnerability/data/death_data/summaries/combined_sum_2008_2012.csv

Processing 2013_2017...
Saved → /Users/larasch/Documents/UCB_postdoc/Research/Mexico_heat_vulnerability/data/death_data/summaries/combined_sum_2013_2017.csv

Processing 2018_2022...
Saved → /Users/larasch/Documents/UCB_postdoc/Research/Mexico_heat_vulnerability/data/death_data/summaries/combined_sum_2018_2022.csv
